# T22 / E03 — Chunk-aware chia theo câu

**Đây là thí nghiệm mà cả đề tài tồn tại vì nó.**

E02 đã tái lập Lookback Lens và đạt macro-F1 **0,7465** bằng cách hỏi ma trận chú ý đúng một
câu: *bao nhiêu* phần chú ý rơi vào ngữ cảnh. Lần này hỏi câu thứ hai: phần đó **trải trên các
đoạn ngữ cảnh thế nào**.

## Mốc phải vượt là 0,7465, không phải 0,6562

Chunk-aware và lookback gộp là **cùng một họ phương pháp**, chỉ khác cách chia mẫu số. Vượt E01
chỉ chứng minh *chú ý có ích* — E02 đã làm xong việc đó. Vượt **E02** mới chứng minh *chia theo
đoạn có ích*, mà đó mới là đóng góp.

Muốn nói **hơn hẳn** thì phải vượt **0,7773**, cận trên khoảng tin cậy của E02.

## Lượt này rẻ hơn T20 dù làm nhiều hơn

E02 và E03 **dùng chung một lượt trích GPU**: chúng chạy cùng mô hình đọc, cùng dữ liệu, cùng
cách chia đoạn, chỉ khác ở chỗ đưa đặc trưng nào cho bộ phân loại — mà đó là quyết định sau khi
GPU đã xong. Tên file đặc trưng vì thế băm theo **phần quyết định lượt chạy GPU**, không băm cả
cấu hình.

| Việc | Số mẫu | Ước tính |
|---|---|---|
| Trích tập train | 5.600 | ~43 phút |
| Trích tập dev | 700 | ~5 phút |
| Trích tập test | 700 | ~5 phút |
| Chọn cách gộp đầu + chấm điểm | — | vài phút, chạy CPU |

**Tổng khoảng 55 phút.** Phải trích lại vì lượt T20 chưa lưu mảng theo đoạn — T20 không cần nên
không lưu. Từ lượt này trở đi thì E04, E05, E12 đều dùng lại được phần đã trích.

Lần này có thêm **tập dev**, vì cách gộp đầu chú ý phải chọn trên dev chứ không phải trên test.

## Notebook settings

- Accelerator: **GPU T4 x2**
- Internet: **On**
- Data: attach dataset `unicorn1209/vihallulens`

## Chuẩn bị

Ba ô dưới đây giống notebook T20. Khoảng 2 phút.

In [ ]:
# Ô 1 — lấy code. Chạy lại được nhiều lần.
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wsunicorn/vihallulens.git"
REPO_DIR = Path("/kaggle/working/vihallulens")


def run(*args, cwd=None):
    done = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(" ".join(args) + chr(10) + done.stdout + done.stderr)
    return done.stdout.strip()


if (REPO_DIR / ".git").is_dir():
    run("git", "fetch", "--quiet", "origin", cwd=REPO_DIR)
    run("git", "reset", "--quiet", "--hard", "origin/main", cwd=REPO_DIR)
    print("đã cập nhật repo có sẵn")
else:
    run("git", "clone", "--quiet", REPO_URL, str(REPO_DIR))
    print("đã clone mới")

%cd /kaggle/working/vihallulens
print("commit:", run("git", "log", "--oneline", "-1", cwd=REPO_DIR))

In [ ]:
# Ô 2 — cài đặt. bitsandbytes cần cho lượng tử hóa 4 bit.
# hình đọc 7B. Không có nó thì mô hình phải nạp ở float16 và tràn 16 GB.
!pip install -q --no-deps -e .
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
# Ô 3 — chuẩn bị dữ liệu và kiểm tra môi trường. Khoảng 1 phút, chạy CPU.
# Bộ kiểm thử chạy trước khi tốn GPU: nó bắt được lỗi công thức mà không cần card.
get_ipython().system("python scripts/probe_env.py")
get_ipython().system("python scripts/normalize_data.py --dataset vihallu")
get_ipython().system("python scripts/split_data.py --only vihallu")
get_ipython().system(
    "python -m pytest tests/test_chunk_features.py tests/test_assemble.py"
    " tests/test_lookback.py tests/test_splits.py -q"
)

## Trích đặc trưng

Ba ô, tập train trước vì nó dài nhất. Mỗi ô **chạy lại được**: mẫu nào tính xong ghi xuống ngay,
chạy lại thì bỏ qua phần đã có.

**Đọc gì trong lúc chạy:** cột `lỗi` phải là 0, và dòng `khối đặc trưng ghi ra` phải liệt kê đủ
bảy khối — hai khối lookback và năm khối chunk-aware. Thiếu khối nào thì ô 7 sẽ báo lỗi rõ ràng
chứ không âm thầm chạy với ma trận hẹp hơn.

In [ ]:
# Ô 4 — trích đặc trưng tập train. Khoảng 43 phút.
!python scripts/extract_features.py --config configs/e03_chunk_sentence_vihallu.yaml --split train

In [ ]:
# Ô 5 — trích tập dev. Khoảng 5 phút. Tập này để CHỌN cách gộp đầu chú ý, không để báo kết quả.
!python scripts/extract_features.py --config configs/e03_chunk_sentence_vihallu.yaml --split dev

In [ ]:
# Ô 6 — trích tập test. Khoảng 5 phút.
!python scripts/extract_features.py --config configs/e03_chunk_sentence_vihallu.yaml --split test

In [ ]:
# Ô 7 — chọn cách gộp đầu trên dev, rồi chấm test đúng một lần. Vài phút, chạy CPU.
# Copy toàn bộ output.
!python scripts/run_chunk_aware.py --config configs/e03_chunk_sentence_vihallu.yaml

In [ ]:
# Ô 8 — chạy lại E02 trên chính lượt trích này, để so sánh không lẫn biến nào khác.
# Vài giây. Con số phải khớp 0,7465 của lượt T20; lệch nhiều nghĩa là có gì đó đã đổi.
!python scripts/run_lookback_baseline.py --config configs/e02_lookback_vihallu.yaml

In [ ]:
# Ô 9 — lấy kết quả và đặc trưng thô về. Đặc trưng lần này dùng lại được cho E04, E05, E12.
import shutil
from pathlib import Path

shutil.copy("results/runs.jsonl", "/kaggle/working/runs.jsonl")
for path in sorted(Path("data/processed").glob("*.jsonl")):
    shutil.copy(path, f"/kaggle/working/{path.name}")
    print(f"{path.name}  {path.stat().st_size / 1024**2:.1f} MB")

with open("results/runs.jsonl", encoding="utf-8") as handle:
    print(handle.read())

## Đọc kết quả thế nào

**Câu hỏi duy nhất: macro-F1 có vượt 0,7465 của E02 không.** Script tự chấm và tự nói ở cuối.

Không vượt thì **đừng trình bày như một cải tiến**. Script sẽ nói thẳng là chưa vượt và trả mã
lỗi. Lúc đó phải kiểm cách chia đoạn và cách gộp đầu trước, rồi mới kết luận — chứ không phải
kết luận rằng chia theo đoạn vô dụng.

Bốn thứ cần nhìn:

1. **Bảng chọn cách gộp đầu**, in trước điểm số. Nếu `all` với 4.536 chiều thắng trên dev thì
   4.536 chiều trên 5.600 mẫu là tỷ lệ đáng ngờ — nhìn xem `topk_heads` kém hơn bao nhiêu, vì
   một mô hình hẹp hơn mà chỉ kém chút xíu thì đáng tin hơn.

2. **Trọng số chia theo khối đặc trưng.** Nếu `lookback_total` chiếm gần hết thì bộ phân loại
   thực chất đang dùng lại E02 và năm đặc trưng mới không đóng góp gì — điểm số có thể vẫn nhỉnh
   hơn do nhiễu. Đây là phép kiểm quan trọng nhất về **cơ chế**, tách khỏi phép kiểm về điểm số.

3. **Tỷ lệ ngữ cảnh chỉ có một đoạn.** Ở đó chunk-aware thoái hóa thành lookback gộp, nên nếu tỷ
   lệ này cao thì phần cải thiện chỉ có thể đến từ phần mẫu còn lại.

4. **F1 lớp `intrinsic`.** E02 đạt 0,7308 — cao nhất bảng, vượt cả PhoBERT. Nếu chunk-aware đúng
   như giả thuyết thì đây là chỗ nó phải cải thiện tiếp, vì cơ chế của nó nhắm thẳng vào đó.

## Ô 8 để làm gì

Chạy lại E02 trên **chính lượt trích này**. Hai thí nghiệm khi đó khác nhau đúng một biến là
nhóm đặc trưng — không lẫn khác biệt nào từ một lượt chạy GPU khác. Con số phải khớp 0,7465 của
lượt T20; lệch nhiều nghĩa là có gì đó đã đổi ngoài ý muốn.